# 13 — Fragrantica–Scento Enrichment

## Objective
Prepare Fragrantica and Scento records, reconcile brand names, identify fragrance-level matches, review ambiguous matches, and enrich the Fragrantica dataset without losing or collapsing source records.

## Matching principles
- Fragrantica is the core reference: one output row per Fragrantica `fragrance_id`.
- Preserve original source fields and Fragrantica record identifiers.
- Use URL-derived fragrance names and normalized brand/name keys for matching. Do not use the concatenated Fragrantica `Name` field as a matching key; it combines fragrance, brand, and gender text.
- Keep Fragrantica gender variants and distinct IDs separate.
- Use only exact brand matches and the 12 user-confirmed aliases. Balmain → Pierre-Balmain remains under review; other unmatched Scento houses remain unmatched in this dataset.
- Do not force ambiguous product matches into the enriched table. Retain them in a review table.
- Keep Scento attributes under `scento_` names so they remain distinguishable from Fragrantica accords and fields.

## Sources
- `main.fragrance_analytics.fragrantica_perfumes`
- `main.fragrance_analytics.scento_perfumes`
- `main.fragrance_analytics.scento_houses`

Scento materials are not joined here because a validated fragrance-to-material relationship has not yet been established.

## Outputs
- `main.fragrance_analytics.fragrantica_prepared`
- `main.fragrance_analytics.scento_prepared`
- `main.fragrance_analytics.scento_fragrantica_brand_mapping`
- `main.fragrance_analytics.fragrantica_scento_match_candidates`
- `main.fragrance_analytics.fragrantica_scento_match_review`
- `main.fragrance_analytics.fragrantica_scento_enriched`

Run the notebook from top to bottom. The checks are designed to stop execution if a supposedly unique key or output grain is violated.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window

# Load the source tables. Source inspection was completed during the initial profiling;
# this notebook starts with preparation and reconciliation.
df_perfumes = spark.table("main.fragrance_analytics.fragrantica_perfumes")
df_scento_perfumes = spark.table("main.fragrance_analytics.scento_perfumes")
df_scento_houses = spark.table("main.fragrance_analytics.scento_houses")

# Shared normalization helper for brand and fragrance matching keys.
# It standardizes common accented Latin characters, punctuation, case, and whitespace.
accented_chars = "ÀÁÂÃÄÅàáâãäåÇçÈÉÊËèéêëÌÍÎÏìíîïÑñÒÓÔÕÖØòóôõöøÙÚÛÜùúûüÝŸýÿ"
plain_chars    = "AAAAAAaaaaaaCcEEEEeeeeIIIIiiiiNnOOOOOOooooooUUUUuuuuYYyy"

def normalize_key(column_name):
    return F.lower(
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.translate(F.col(column_name), accented_chars, plain_chars),
                    r"[^\w\s]", " "
                ),
                r"\s+", " "
            )
        )
    )

df_fragrantica_prepared = (
    df_perfumes
    # Standardize source column names while preserving the original Name field.
    .withColumnRenamed("Gender", "gender")
    .withColumnRenamed("Rating Value", "rating_value")
    .withColumnRenamed("Rating Count", "rating_count")
    .withColumnRenamed("Main Accords", "main_accords")
    .withColumnRenamed("Perfumers", "perfumers")
    .withColumnRenamed("Description", "description")

    # Extract brand and fragrance identifiers from the URL.
    .withColumn(
        "brand_slug",
        F.regexp_extract("url", r"/perfume/([^/]+)/", 1)
    )
    .withColumn(
        "fragrance_slug",
        F.regexp_extract("url", r"/perfume/[^/]+/(.*?)-[0-9]+\.html", 1)
    )
    .withColumn(
        "fragrance_id",
        F.regexp_extract("url", r"-([0-9]+)\.html(?:\?.*)?$", 1)
    )

    # Canonical brand name derived from the URL.
    .withColumn(
        "brand_clean",
        F.initcap(F.regexp_replace(F.col("brand_slug"), "-", " "))
    )
    .withColumn("brand_name", F.col("brand_clean"))

    # Remove the gender suffix from the original concatenated Name field.
    # The original Name remains unchanged.
    .withColumn(
        "name_without_gender",
        F.trim(
            F.regexp_replace(
                F.col("Name"),
                r"(?i)for\s+(women\s+and\s+men|men\s+and\s+women|women|men)$",
                ""
            )
        )
    )

    # Remove the brand suffix using its length, based on the URL-derived brand.
    # This preserves punctuation in fragrance names, e.g. J'ose and 1 Million $.
    .withColumn(
        "fragrance_name_candidate",
        F.trim(
            F.expr(
                """
                substring(
                    name_without_gender,
                    1,
                    length(name_without_gender) - length(brand_clean)
                )
                """
            )
        )
    )

    # Fallback to the URL slug only if extraction from Name is unavailable.
    .withColumn(
        "fragrance_name_candidate",
        F.when(
            F.col("fragrance_name_candidate").isNull()
            | (F.col("fragrance_name_candidate") == ""),
            F.regexp_replace(F.col("fragrance_slug"), "-", " ")
        ).otherwise(F.col("fragrance_name_candidate"))
    )

    .withColumn(
        "fragrance_name_clean",
        F.trim(F.col("fragrance_name_candidate"))
    )

    # Matching keys are normalized separately from display names.
    .withColumn("brand_match_key", normalize_key("brand_clean"))
    .withColumn("fragrance_match_key", normalize_key("fragrance_name_clean"))

    # Remove the temporary parsing column.
    .drop("name_without_gender")
)

# Keep source records intact; these checks validate the intended Fragrantica grain.
fragrantica_total = df_fragrantica_prepared.count()

fragrantica_id_stats = (
    df_fragrantica_prepared
    .agg(
        F.countDistinct("fragrance_id").alias("unique_fragrance_ids"),
        F.sum(
            F.when(
                F.col("fragrance_id").isNull() | (F.col("fragrance_id") == ""),
                1
            ).otherwise(0)
        ).alias("missing_ids"),
        F.sum(
            F.when(
                F.col("brand_match_key").isNull() | (F.col("brand_match_key") == ""),
                1
            ).otherwise(0)
        ).alias("missing_brand_keys"),
        F.sum(
            F.when(
                F.col("fragrance_match_key").isNull() | (F.col("fragrance_match_key") == ""),
                1
            ).otherwise(0)
        ).alias("missing_fragrance_keys")
    )
)

display(fragrantica_id_stats)

assert fragrantica_total == df_fragrantica_prepared.select("fragrance_id").distinct().count(), \
    "Fragrantica source is not one row per fragrance_id; investigate before proceeding."


In [0]:
df_scento_prepared = (
    df_scento_perfumes
    .withColumn("house_match_key", normalize_key("house"))
    .withColumn("fragrance_match_key", normalize_key("name"))
)

df_scento_houses_prepared = (
    df_scento_houses
    .withColumn("house_match_key", normalize_key("name"))
)

scento_checks = df_scento_prepared.agg(
    F.count("*").alias("total_perfume_records"),
    F.countDistinct("id").alias("unique_perfume_ids"),
    F.countDistinct("house_match_key").alias("unique_house_keys"),
    F.sum(F.when(F.col("house_match_key").isNull() | (F.col("house_match_key") == ""), 1).otherwise(0)).alias("missing_house_keys"),
    F.sum(F.when(F.col("fragrance_match_key").isNull() | (F.col("fragrance_match_key") == ""), 1).otherwise(0)).alias("missing_fragrance_keys")
)
display(scento_checks)

assert df_scento_prepared.count() == df_scento_prepared.select("id").distinct().count(), \
    "Scento perfume IDs are not unique; investigate before matching."
assert df_scento_houses_prepared.count() == df_scento_houses_prepared.select("house_match_key").distinct().count(), \
    "Scento house keys are not unique; investigate before brand mapping."


In [0]:
# One distinct Fragrantica brand key per normalized key.
df_fragrantica_brand_keys = (
    df_fragrantica_prepared
    .select("brand_match_key")
    .filter(F.col("brand_match_key").isNotNull() & (F.col("brand_match_key") != ""))
    .distinct()
)

# Exact brand-key matches.
df_exact_brand_mapping = (
    df_scento_houses_prepared.alias("s")
    .join(
        df_fragrantica_brand_keys.alias("f"),
        F.col("s.house_match_key") == F.col("f.brand_match_key"),
        "inner"
    )
    .select(
        F.col("s.name").alias("scento_house"),
        F.col("s.house_match_key").alias("scento_house_key"),
        F.col("f.brand_match_key").alias("fragrantica_brand_key"),
        F.lit("EXACT_MATCH").alias("mapping_status")
    )
)

# User-confirmed aliases only. The target values are the Fragrantica URL brand slugs
# represented in the current dataset, not a reconstruction of today's website taxonomy.
confirmed_aliases = [
    ("Al Haramain", "Al-Haramain-Perfumes"),
    ("Annick Goutal", "Goutal"),
    ("Dunhill", "Alfred-Dunhill"),
    ("Floris London", "Floris"),
    ("Kayali", "Kayali-Fragrances"),
    ("Kilian Paris", "By-Kilian"),
    ("Lacoste", "Lacoste-Fragrances"),
    ("Lattafa", "Lattafa-Perfumes"),
    ("Maison Margiela Fragrances", "Maison-Martin-Margiela"),
    ("Nicolaï", "Nicolai-Parfumeur-Createur"),
    ("Rabanne", "Paco-Rabanne"),
    ("Roja Parfums", "Roja-Dove")
]

df_alias_input = spark.createDataFrame(
    confirmed_aliases,
    ["scento_house", "fragrantica_brand_slug"]
)

df_confirmed_alias_mapping = (
    df_alias_input
    .withColumn("scento_house_key", normalize_key("scento_house"))
    .withColumn("fragrantica_brand_key", normalize_key("fragrantica_brand_slug"))
    .select(
        "scento_house",
        "scento_house_key",
        "fragrantica_brand_key",
        F.lit("CONFIRMED_ALIAS").alias("mapping_status")
    )
)

# Balmain was discussed but not confirmed. Keep it visible in the audit, not in matching.
df_review_brand = (
    spark.createDataFrame(
        [("Balmain", "Pierre-Balmain")],
        ["scento_house", "fragrantica_brand_slug"]
    )
    .withColumn("scento_house_key", normalize_key("scento_house"))
    .withColumn("fragrantica_brand_key", normalize_key("fragrantica_brand_slug"))
    .select(
        "scento_house",
        "scento_house_key",
        "fragrantica_brand_key",
        F.lit("REVIEW_REQUIRED").alias("mapping_status")
    )
)

df_confirmed_brand_mapping = (
    df_exact_brand_mapping
    .unionByName(df_confirmed_alias_mapping)
    .dropDuplicates(["scento_house_key"])
)

# Ensure that no Scento house is mapped to multiple Fragrantica brand keys.
duplicate_confirmed_brand_keys = (
    df_confirmed_brand_mapping
    .groupBy("scento_house_key")
    .agg(F.countDistinct("fragrantica_brand_key").alias("target_count"))
    .filter(F.col("target_count") > 1)
)
assert duplicate_confirmed_brand_keys.count() == 0, \
    "A Scento house maps to multiple Fragrantica brand keys."

# Full 160-house audit: exact, confirmed alias, review required, or unmatched.
df_brand_mapping_audit = (
    df_scento_houses_prepared
    .select(
        F.col("name").alias("scento_house"),
        F.col("house_match_key").alias("scento_house_key")
    )
    .join(
        df_confirmed_brand_mapping.select(
            "scento_house_key", "fragrantica_brand_key", "mapping_status"
        ),
        "scento_house_key",
        "left"
    )
    .join(
        df_review_brand.select(
            "scento_house_key",
            F.col("fragrantica_brand_key").alias("review_fragrantica_brand_key"),
            F.col("mapping_status").alias("review_status")
        ),
        "scento_house_key",
        "left"
    )
    .withColumn(
        "fragrantica_brand_key",
        F.coalesce("fragrantica_brand_key", "review_fragrantica_brand_key")
    )
    .withColumn(
        "mapping_status",
        F.coalesce("mapping_status", "review_status", F.lit("UNMATCHED"))
    )
    .drop("review_fragrantica_brand_key", "review_status")
)

display(
    df_brand_mapping_audit
    .groupBy("mapping_status")
    .agg(
        F.count("*").alias("house_rows"),
        F.countDistinct("scento_house_key").alias("unique_scento_houses")
    )
    .orderBy("mapping_status")
)

assert df_brand_mapping_audit.count() == 160, "Brand mapping audit should contain all 160 Scento houses."

In [0]:
df_scento_with_brand_mapping = (
    df_scento_prepared.alias("s")
    .join(
        df_confirmed_brand_mapping.select(
            "scento_house_key",
            "fragrantica_brand_key",
            "mapping_status"
        ).alias("m"),
        F.col("s.house_match_key") == F.col("m.scento_house_key"),
        "inner"
    )
)

df_enrichment_candidates = (
    df_scento_with_brand_mapping.alias("s")
    .join(
        df_fragrantica_prepared.alias("f"),
        (F.col("s.fragrantica_brand_key") == F.col("f.brand_match_key")) &
        (F.col("s.fragrance_match_key") == F.col("f.fragrance_match_key")),
        "inner"
    )
    .select(
        "f.*",
        F.col("s.id").alias("scento_id"),
        F.col("s.name").alias("scento_name"),
        F.col("s.house").alias("scento_house"),
        F.col("s.year").alias("scento_year"),
        F.col("s.perfumers").alias("scento_perfumers"),
        F.col("s.concentration").alias("scento_concentration"),
        F.col("s.family").alias("scento_family"),
        F.col("s.secondary_family").alias("scento_secondary_family"),
        F.col("s.character").alias("scento_character"),
        F.col("s.seasons").alias("scento_seasons"),
        F.col("s.times_of_day").alias("scento_times_of_day"),
        F.col("s.top_notes").alias("scento_top_notes"),
        F.col("s.heart_notes").alias("scento_heart_notes"),
        F.col("s.base_notes").alias("scento_base_notes"),
        F.col("s.notes").alias("scento_notes"),
        F.col("s.description").alias("scento_description"),
        F.col("s.popularity_rank").alias("scento_popularity_rank"),
        F.col("s.url").alias("scento_url"),
        F.col("s.mapping_status").alias("brand_mapping_status")
    )
    .dropDuplicates(["fragrance_id", "scento_id"])
)

candidate_summary = df_enrichment_candidates.agg(
    F.count("*").alias("candidate_pairs"),
    F.countDistinct("fragrance_id").alias("fragrantica_fragrances"),
    F.countDistinct("scento_id").alias("scento_perfumes")
)
display(candidate_summary)

In [0]:
scento_match_counts = (
    df_enrichment_candidates
    .groupBy("scento_id")
    .agg(F.countDistinct("fragrance_id").alias("fragrantica_match_count"))
)

fragrantica_match_counts = (
    df_enrichment_candidates
    .groupBy("fragrance_id")
    .agg(F.countDistinct("scento_id").alias("scento_match_count"))
)

df_enrichment_candidates = (
    df_enrichment_candidates
    .join(scento_match_counts, "scento_id", "left")
    .join(fragrantica_match_counts, "fragrance_id", "left")
    .withColumn(
        "match_status",
        F.when(F.col("fragrantica_match_count") > 1, "AMBIGUOUS_REVIEW")
         .when(F.col("scento_match_count") > 1, "AMBIGUOUS_REVIEW")
         .otherwise("UNIQUE_EXACT_MATCH")
    )
)

df_match_review = (
    df_enrichment_candidates
    .filter(F.col("match_status") == "AMBIGUOUS_REVIEW")
    .orderBy("scento_house", "scento_name", "gender", "fragrance_id")
)

display(
    df_enrichment_candidates
    .groupBy("match_status")
    .agg(
        F.count("*").alias("candidate_pairs"),
        F.countDistinct("fragrance_id").alias("fragrantica_fragrances"),
        F.countDistinct("scento_id").alias("scento_perfumes")
    )
)

display(
    df_match_review.select(
        "scento_id", "scento_name", "scento_house", "scento_concentration",
        "fragrance_id", "Name", "gender", "fragrance_name_clean",
        "rating_value", "rating_count", "main_accords", "url", "scento_url",
        "fragrantica_match_count", "scento_match_count", "match_status"
    )
)

In [0]:
df_approved_matches = (
    df_enrichment_candidates
    .filter(F.col("match_status") == "UNIQUE_EXACT_MATCH")
)

# The approved set must be one-to-one at both source identifiers.
assert df_approved_matches.count() == df_approved_matches.select("fragrance_id").distinct().count(), \
    "Approved matches contain multiple Scento rows per Fragrantica ID."
assert df_approved_matches.count() == df_approved_matches.select("scento_id").distinct().count(), \
    "Approved matches contain multiple Fragrantica rows per Scento ID."

df_review_fragrance_ids = (
    df_match_review
    .select("fragrance_id")
    .distinct()
    .withColumn("has_ambiguous_candidate", F.lit(True))
)

scento_enrichment_columns = [
    "scento_id", "scento_name", "scento_house", "scento_year",
    "scento_perfumers", "scento_concentration", "scento_family",
    "scento_secondary_family", "scento_character", "scento_seasons",
    "scento_times_of_day", "scento_top_notes", "scento_heart_notes",
    "scento_base_notes", "scento_notes", "scento_description",
    "scento_popularity_rank", "scento_url", "brand_mapping_status"
]

df_fragrantica_scento_enriched = (
    df_fragrantica_prepared.alias("f")
    .join(
        df_approved_matches.select("fragrance_id", *scento_enrichment_columns).alias("s"),
        "fragrance_id",
        "left"
    )
    .join(df_review_fragrance_ids, "fragrance_id", "left")
    .withColumn(
        "scento_match_status",
        F.when(F.col("scento_id").isNotNull(), "MATCHED_UNIQUE")
         .when(F.col("has_ambiguous_candidate") == True, "AMBIGUOUS_REVIEW")
         .otherwise("NO_MATCH")
    )
    .drop("has_ambiguous_candidate")
)

display(
    df_fragrantica_scento_enriched
    .groupBy("scento_match_status")
    .agg(
        F.count("*").alias("fragrantica_rows"),
        F.countDistinct("fragrance_id").alias("unique_fragrance_ids")
    )
    .orderBy("scento_match_status")
)

assert df_fragrantica_scento_enriched.count() == fragrantica_total, \
    "Enriched table row count differs from the prepared Fragrantica source."
assert df_fragrantica_scento_enriched.count() == df_fragrantica_scento_enriched.select("fragrance_id").distinct().count(), \
    "Enriched table is not one row per Fragrantica fragrance_id."

In [0]:
# Fragrantica population and row-grain validation.
final_checks = df_fragrantica_scento_enriched.agg(
    F.count("*").alias("enriched_rows"),
    F.countDistinct("fragrance_id").alias("unique_fragrance_ids"),
    F.sum(F.when(F.col("fragrance_id").isNull() | (F.col("fragrance_id") == ""), 1).otherwise(0)).alias("missing_fragrance_ids"),
    F.sum(F.when(F.col("scento_match_status") == "MATCHED_UNIQUE", 1).otherwise(0)).alias("uniquely_matched_rows"),
    F.sum(F.when(F.col("scento_match_status") == "AMBIGUOUS_REVIEW", 1).otherwise(0)).alias("ambiguous_review_rows"),
    F.sum(F.when(F.col("scento_match_status") == "NO_MATCH", 1).otherwise(0)).alias("unmatched_rows")
)
display(final_checks)

# Candidate-level ambiguity diagnostics.
display(
    df_enrichment_candidates
    .groupBy("scento_id", "scento_name", "scento_house")
    .agg(
        F.countDistinct("fragrance_id").alias("fragrantica_records"),
        F.collect_set("gender").alias("fragrantica_genders"),
        F.collect_set("fragrance_id").alias("fragrance_ids")
    )
    .filter(F.col("fragrantica_records") > 1)
    .orderBy("scento_house", "scento_name")
)

assert df_fragrantica_scento_enriched.count() == fragrantica_total
assert df_fragrantica_scento_enriched.select("fragrance_id").distinct().count() == fragrantica_total

In [0]:
(
    df_fragrantica_prepared
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_prepared")
)

(
    df_scento_prepared
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.scento_prepared")
)

(
    df_brand_mapping_audit
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.scento_fragrantica_brand_mapping")
)

(
    df_enrichment_candidates
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_scento_match_candidates")
)

(
    df_match_review
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_scento_match_review")
)

(
    df_fragrantica_scento_enriched
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_scento_enriched")
)

print("Notebook 13 outputs saved successfully.")

In [0]:
display(
    df_brand_audit
    .filter(
        (F.col("brand_suffix_valid") == False)
        | F.col("brand_suffix_valid").isNull()
    )
    .select(
        "fragrance_id",
        "Name",
        "brand_slug",
        "brand_name",
        "name_without_gender",
        "source_name_key",
        "brand_suffix_key",
        "url"
    )
    .orderBy("brand_name", "Name")
    .limit(50)
)